# Hybrid RAG Building 

In [22]:
import warnings 
warnings.filterwarnings('ignore')

from glob import glob
from langchain_community.document_loaders import PyPDFLoader

pdf_files = glob("data/research_paper/*.pdf")

documents = []

for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    documents.extend(loader.load())

print(f"PDFs loaded: {len(pdf_files)}")
print(f"Pages loaded: {len(documents)}")

PDFs loaded: 3
Pages loaded: 11


In [23]:
# Create chunks 
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=120
)

texts = text_splitter.split_documents(documents)

chunks = [i.page_content for i in texts]
print(f"the len of chunks : {len(chunks)}")

the len of chunks : 20


In [24]:
# Create chromaDB
import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
embedding_function = SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-small-en-v1.5")
client = chromadb.PersistentClient(path="./Local_data")
collection = client.get_or_create_collection(name="edumind",embedding_function=embedding_function)
if collection.count()==0:
    collection.add(
        documents=chunks,
        ids=[str(i) for i in range(len(chunks))],
        metadatas=[i.metadata for i in texts]
    )
collection.count()

20

In [25]:
# corpus create 
from rank_bm25 import BM25Okapi
def fun(tokens):
    tokens = tokens.lower()
    tokens = tokens.split()
    return tokens
try:
    t_corpus=BM25Okapi([fun(i) for i in chunks])
    print(f"suceessfully : {t_corpus}") 
except Exception as e :
    print(str(e))

suceessfully : <rank_bm25.BM25Okapi object at 0x12ec42360>


In [26]:
# Import LLM 
from langchain_groq import ChatGroq 
import os 
from dotenv import load_dotenv 
load_dotenv()
key = os.getenv("GROQ_API_KEY")
groq = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
from tavily import TavilyClient
def Hybrid_search(query:str , n_near_chunks=3):
    # vectordb symentic search 
    query_rewrite = groq.invoke(query).content 
    result = collection.query(query_texts=[query_rewrite],n_results=n_near_chunks)
    documents = result['documents'][0]
    distances = result['distances'][0]
    threshold = 1.0 
    fetch_chunks = []
    for i , docs in zip(distances,documents):
        if i < threshold : 
            fetch_chunks.append(docs)
    # Hybrid connetion 
    cor_scores=t_corpus.get_scores(query=query_rewrite.split())
    def find_top_docs(score,k=10):
        index = list(enumerate(score))
        indx_sorted = sorted(index, key=lambda x : x[1],reverse=True)
        return [i for i , doc in indx_sorted[:k]]
    get_docs = find_top_docs(score=cor_scores,k=10)

    copy_get_chunks = [chunks[i] for i in get_docs]

    rrf_token = {}
    for rank , doc in enumerate(fetch_chunks):
        rrf_token[doc] = rrf_token.get(doc,0)+1/(rank+60)
    for rank , doc in enumerate(copy_get_chunks):
        rrf_token[doc] = rrf_token.get(doc,0)+1/(rank+60)
    marged_docs=sorted(rrf_token.items(),key=lambda x:x[1],reverse=True)
    return_near_dos=[i for i , _ in marged_docs[:5]]
    if return_near_dos:
        return return_near_dos 
    client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

    response = client.search(
    query=query_rewrite,
    search_depth="advanced"
    )
    return response

In [37]:
# testing 
query = "can you tell me 3 points about The Effects of Pollution on Marine Life ? "
retrieved_context = Hybrid_search(query)

prompt = f"""
Answer the question using only the context below.

Context:
{retrieved_context}

Question:
{query}

If the answer is not present in the context, say:
"I could not find the answer in the provided documents."
"""

result = groq.invoke(prompt)
print(result.content)

**Three key effects of pollution on marine life**

1. **Plastic waste** – Large pieces and micro‑plastics are ingested or entangle animals such as turtles, fish and seabirds, causing injury, illness and death. The micro‑plastics further contaminate the water and move up the food chain, ultimately impacting human health.

2. **Chemical runoff (pesticides, fertilizers, industrial waste)** – These chemicals cause eutrophication, leading to excessive algal blooms that deplete oxygen in the water. The resulting “dead zones” prevent most marine organisms from surviving and severely reduce biodiversity and the availability of marine resources.

3. **Overall ecosystem damage** – Pollution degrades water quality, harms marine species’ health and habitats, and diminishes the overall biodiversity of ocean ecosystems, threatening both marine life and the communities that depend on them.
